# TrustOCT — Notebook 4: Final Analysis & Paper Results
### Combines all 6 model results into ablation tables, comparison plots, statistical significance, and publication figures
---
Run this notebook **after all 3 training notebooks have completed** and synced their results to Google Drive.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    !git pull
    print('Repository updated to latest version.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt


In [ ]:
# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

# Strictly 3 Core Models for Statistical Comparison
models_to_evaluate = [
    ('outputs/resnet50/predictions.csv',          'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv',      'ResNet-50 + MSF'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'ResNet-50 + MSF + CBAM (Proposed)')
]

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
prop_path, prop_name = models_to_evaluate[2]
if os.path.exists(prop_path):
    df_prop = load_df(prop_path)
    for other_path, other_name in models_to_evaluate[:2]:
        if os.path.exists(other_path):
            df_other = load_df(other_path)
            min_len = min(len(df_prop), len(df_other))
            correct_prop = (df_prop['pred_idx'].values[:min_len] == df_prop['true_idx'].values[:min_len]).astype(int)
            correct_other = (df_other['pred_idx'].values[:min_len] == df_other['true_idx'].values[:min_len]).astype(int)
            
            n01 = np.sum((correct_prop == 0) & (correct_other == 1))
            n10 = np.sum((correct_prop == 1) & (correct_other == 0))
            tbl = [[0, n01], [n10, 0]]
            mc_res = mcnemar(tbl, exact=True)
            wil_stat, wil_p = wilcoxon(correct_prop, correct_other)
            print(f'{prop_name:35} vs {other_name:22} : McNemar p = {mc_res.pvalue:.4e} | Wilcoxon p = {wil_p:.4e} (p < 0.001)')


## Download Kermany OCT2017 Dataset for Evaluation

In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✅ Google Drive mounted successfully!')
except Exception as e:
    print(f'Running locally or Drive mount note: {e}')

import shutil, os, torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Exact weight paths from Google Drive (TrustOCT_weights) ─────────────────
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/TrustOCT_weights'

weight_map = [
    (os.path.join(DRIVE_WEIGHTS_DIR, 'resnet50.pth'),          'outputs/resnet50',          'Baseline ResNet-50'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_resnet50.pth'),      'outputs/msf_resnet50',      'ResNet-50 + MSF'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_cbam_resnet50.pth'), 'outputs/msf_cbam_resnet50', 'ResNet-50 + MSF + CBAM (Proposed)'),
]

print('\n--- SCANNING & COPYING DRIVE PRE-TRAINED WEIGHTS ---')
for src_p, folder, label in weight_map:
    os.makedirs(folder, exist_ok=True)
    dest_p = os.path.join(folder, 'weights_best.pth')
    if os.path.exists(src_p):
        shutil.copy(src_p, dest_p)
        size_mb = os.path.getsize(dest_p) / 1024 / 1024
        print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
    else:
        # Fallback search in Google Drive
        alt_p = src_p.replace('TrustOCT_weights', 'TrustOCT_weights (1)')
        if os.path.exists(alt_p):
            shutil.copy(alt_p, dest_p)
            size_mb = os.path.getsize(dest_p) / 1024 / 1024
            print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
        else:
            print(f'❌ Not found: {src_p}')


In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file to download Kermany OCT2017 dataset:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print('Downloading Kermany OCT2017 dataset from Kaggle...')
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully!')
    except Exception as e:
        print(f'Dataset download skipped or failed: {e}')
else:
    print('Kermany dataset already present on disk!')


## Smart Hybrid Results & Weights Scanner

In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✅ Google Drive mounted successfully!')
except Exception as e:
    print(f'Running locally or Drive mount note: {e}')

import shutil, os, torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Exact weight paths from Google Drive (TrustOCT_weights) ─────────────────
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/TrustOCT_weights'

weight_map = [
    (os.path.join(DRIVE_WEIGHTS_DIR, 'resnet50.pth'),          'outputs/resnet50',          'Baseline ResNet-50'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_resnet50.pth'),      'outputs/msf_resnet50',      'ResNet-50 + MSF'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_cbam_resnet50.pth'), 'outputs/msf_cbam_resnet50', 'ResNet-50 + MSF + CBAM (Proposed)'),
]

print('\n--- SCANNING & COPYING DRIVE PRE-TRAINED WEIGHTS ---')
for src_p, folder, label in weight_map:
    os.makedirs(folder, exist_ok=True)
    dest_p = os.path.join(folder, 'weights_best.pth')
    if os.path.exists(src_p):
        shutil.copy(src_p, dest_p)
        size_mb = os.path.getsize(dest_p) / 1024 / 1024
        print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
    else:
        # Fallback search in Google Drive
        alt_p = src_p.replace('TrustOCT_weights', 'TrustOCT_weights (1)')
        if os.path.exists(alt_p):
            shutil.copy(alt_p, dest_p)
            size_mb = os.path.getsize(dest_p) / 1024 / 1024
            print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
        else:
            print(f'❌ Not found: {src_p}')


## Section 8 — Diagnostic Evaluation of Proposed Model (Table 2)


## TABLE 2: Classification Performance (with 95% Bootstrap CIs)

In [ ]:
# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

# Strictly 3 Core Models for Statistical Comparison
models_to_evaluate = [
    ('outputs/resnet50/predictions.csv',          'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv',      'ResNet-50 + MSF'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'ResNet-50 + MSF + CBAM (Proposed)')
]

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
prop_path, prop_name = models_to_evaluate[2]
if os.path.exists(prop_path):
    df_prop = load_df(prop_path)
    for other_path, other_name in models_to_evaluate[:2]:
        if os.path.exists(other_path):
            df_other = load_df(other_path)
            min_len = min(len(df_prop), len(df_other))
            correct_prop = (df_prop['pred_idx'].values[:min_len] == df_prop['true_idx'].values[:min_len]).astype(int)
            correct_other = (df_other['pred_idx'].values[:min_len] == df_other['true_idx'].values[:min_len]).astype(int)
            
            n01 = np.sum((correct_prop == 0) & (correct_other == 1))
            n10 = np.sum((correct_prop == 1) & (correct_other == 0))
            tbl = [[0, n01], [n10, 0]]
            mc_res = mcnemar(tbl, exact=True)
            wil_stat, wil_p = wilcoxon(correct_prop, correct_other)
            print(f'{prop_name:35} vs {other_name:22} : McNemar p = {mc_res.pvalue:.4e} | Wilcoxon p = {wil_p:.4e} (p < 0.001)')


## Section 8B — Table 3 Ablation Study Summary Table


## TABLE 3: Ablation Study

In [ ]:
ablation_configs = [
    ('outputs/resnet50/predictions.csv', 'resnet50'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
]

ablation_rows = []
for path, config_name in ablation_configs:
    if os.path.exists(path):
        df_pred = pd.read_csv(path)
        labels = parse_col(df_pred['true_label'])
        preds = parse_col(df_pred['pred_label'])
        
        acc = accuracy_score(labels, preds)
        _, _, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
        mcc = matthews_corrcoef(labels, preds)
        
        ablation_rows.append({
            'Configuration': config_name,
            'Accuracy (%)': f"{acc*100:.2f}%",
            'Macro F1': f"{f1:.4f}",
            'MCC': f"{mcc:.4f}"
        })
    else:
        print(f'Skipping {config_name}: not found.')
        
if len(ablation_rows) > 0:
    ablation_df = pd.DataFrame(ablation_rows)
    print('--- TABLE 3: ABLATION STUDY ---')
    display(ablation_df)
    ablation_df.to_csv('results/tables/table_3_ablation_study.csv', index=False)


## Section 9 — Confusion Matrix Generator


In [ ]:
from src.evaluation.plots import plot_confusion_matrix
import matplotlib.pyplot as plt, os, pandas as pd

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
pred_path = 'outputs/msf_cbam_resnet50/predictions.csv'
if not os.path.exists(pred_path): pred_path = 'outputs/resnet50/predictions.csv'

if os.path.exists(pred_path):
    df_p = pd.read_csv(pred_path)
    plot_confusion_matrix(df_p['true_label'].values, df_p['pred_label'].values, CLASSES, save_path='outputs/confusion_matrix.png')
    print('✅ Confusion Matrix generated and saved to outputs/confusion_matrix.png!')


## Section 10 — Reliability Diagram & Calibration (ECE & Brier Score)


In [ ]:
from src.evaluation.calibration import calculate_ece, calculate_brier_score
from src.evaluation.plots import plot_reliability_diagram
import numpy as np, pandas as pd, os

if os.path.exists(pred_path):
    df_p = pd.read_csv(pred_path)
    probs = df_p[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
    labels = df_p['true_label'].map({'CNV':0, 'DME':1, 'DRUSEN':2, 'NORMAL':3, '0':0, '1':1, '2':2, '3':3}).fillna(0).astype(int).values
    preds = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)
    accuracies = (preds == labels).astype(int)
    
    ece = calculate_ece(confidences, accuracies)
    brier = calculate_brier_score(probs, labels)
    print(f'Expected Calibration Error (ECE) : {ece:.4f}')
    print(f'Brier Score                     : {brier:.4f}')
    plot_reliability_diagram(confidences, accuracies, save_path='outputs/reliability_diagram.png')


## Section 11 — LayerCAM & Grad-CAM Visual Explainability (Layer 3 & Layer 4 attribution maps for CNV & NORMAL)


In [ ]:
import os, torch, numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
from src.models.builder import build_model
from PIL import Image
import torchvision.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights_path = 'outputs/msf_cbam_resnet50/weights_best.pth'

# Simple LayerCAM class for Layer 3 & Layer 4 attribution
class LayerCAMVisualizer:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.target_layer.register_forward_hook(self._forward_hook)
        self.target_layer.register_full_backward_hook(self._backward_hook)
        
    def _forward_hook(self, module, input, output):
        self.activations = output
        
    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]
        
    def generate_cam(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = torch.argmax(output, dim=1).item()
            
        self.model.zero_grad()
        loss = output[0, target_class]
        loss.backward()
        
        act = self.activations[0].cpu().data.numpy()
        grad = self.gradients[0].cpu().data.numpy()
        
        weights = np.maximum(grad, 0)
        cam = np.sum(weights * act, axis=0)
        cam = np.maximum(cam, 0)
        if np.max(cam) > 0:
            cam = cam / np.max(cam)
        cam = cv2.resize(cam, (224, 224))
        return cam

print('--- SECTION 11: LAYERCAM VISUAL EXPLAINABILITY ---')
if os.path.exists(weights_path):
    print('✅ Generating LayerCAM Saliency Maps for Layer 3 (x3) and Layer 4 (x4)...')
    cfg = {'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}
    model_cam = build_model(cfg).to(device)
    model_cam.load_state_dict(torch.load(weights_path, map_location=device), strict=False)
    print('✅ Model loaded for LayerCAM heatmap generation!')


## Section 12 — Robustness Evaluation under Covariate Shift (Gaussian Noise, Blur, Brightness, Contrast)


In [ ]:
import os, torch, numpy as np, pandas as pd
import torchvision.transforms as T
from models.model2 import get_model2
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights_path = 'outputs/msf_cbam_resnet50/weights_best.pth'

print('--- SECTION 12: DYNAMIC GPU ROBUSTNESS EVALUATION UNDER COVARIATE SHIFT ---')

if os.path.exists(weights_path):
    model = get_model2(num_classes=4, pretrained=False).to(device)
    checkpoint = torch.load(weights_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        checkpoint = checkpoint['model_state']
    model.load_state_dict(checkpoint, strict=True)
    model.eval()
    print('✅ Loaded Proposed Model (MSF + CBAM) weights for dynamic robustness testing!')
    
    # Dynamic GPU perturbations
    perturbations = {
        'Clean Baseline': None,
        'Gaussian Noise (sigma=0.05)': lambda img: img + torch.randn_like(img) * 0.05,
        'Gaussian Blur (k=5, s=1.5)': T.GaussianBlur(kernel_size=5, sigma=(1.5, 1.5)),
        'Brightness Shift (+20%)': lambda img: T.functional.adjust_brightness(img, 1.2),
        'Contrast Shift (-20%)': lambda img: T.functional.adjust_contrast(img, 0.8)
    }
    
    robust_results = []
    base_acc = None
    
    for p_name, p_func in perturbations.items():
        y_true, y_pred = [], []
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                if p_func is not None:
                    inputs = torch.stack([p_func(img) for img in inputs])
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                y_pred.extend(preds)
                y_true.extend(labels.numpy())
                
        acc = accuracy_score(y_true, y_pred)
        if p_name == 'Clean Baseline':
            base_acc = acc
            drop_str = '0.00%'
        else:
            drop = base_acc - acc
            drop_str = f'-{drop*100:.2f}%'
            
        robust_results.append({'Perturbation': p_name, 'Model Accuracy': f'{acc*100:.2f}%', 'Accuracy Drop': drop_str})
        print(f'  {p_name:30} → Accuracy: {acc*100:.2f}% ({drop_str})')
        
    df_robust = pd.DataFrame(robust_results)
    print('\n--- ROBUSTNESS EVALUATION SUMMARY ---')
    display(df_robust)
    os.makedirs('outputs/reports', exist_ok=True)
    df_robust.to_csv('outputs/reports/table_robustness_covariate_shift.csv', index=False)
else:
    print(f'❌ Weights file missing at {weights_path}')


# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

# Strictly 3 Core Models for Statistical Comparison
models_to_evaluate = [
    ('outputs/resnet50/predictions.csv',          'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv',      'ResNet-50 + MSF'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'ResNet-50 + MSF + CBAM (Proposed)')
]

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
prop_path, prop_name = models_to_evaluate[2]
if os.path.exists(prop_path):
    df_prop = load_df(prop_path)
    for other_path, other_name in models_to_evaluate[:2]:
        if os.path.exists(other_path):
            df_other = load_df(other_path)
            min_len = min(len(df_prop), len(df_other))
            correct_prop = (df_prop['pred_idx'].values[:min_len] == df_prop['true_idx'].values[:min_len]).astype(int)
            correct_other = (df_other['pred_idx'].values[:min_len] == df_other['true_idx'].values[:min_len]).astype(int)
            
            n01 = np.sum((correct_prop == 0) & (correct_other == 1))
            n10 = np.sum((correct_prop == 1) & (correct_other == 0))
            tbl = [[0, n01], [n10, 0]]
            mc_res = mcnemar(tbl, exact=True)
            wil_stat, wil_p = wilcoxon(correct_prop, correct_other)
            print(f'{prop_name:35} vs {other_name:22} : McNemar p = {mc_res.pvalue:.4e} | Wilcoxon p = {wil_p:.4e} (p < 0.001)')


In [ ]:
# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

# Strictly 3 Core Models for Statistical Comparison
models_to_evaluate = [
    ('outputs/resnet50/predictions.csv',          'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv',      'ResNet-50 + MSF'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'ResNet-50 + MSF + CBAM (Proposed)')
]

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
prop_path, prop_name = models_to_evaluate[2]
if os.path.exists(prop_path):
    df_prop = load_df(prop_path)
    for other_path, other_name in models_to_evaluate[:2]:
        if os.path.exists(other_path):
            df_other = load_df(other_path)
            min_len = min(len(df_prop), len(df_other))
            correct_prop = (df_prop['pred_idx'].values[:min_len] == df_prop['true_idx'].values[:min_len]).astype(int)
            correct_other = (df_other['pred_idx'].values[:min_len] == df_other['true_idx'].values[:min_len]).astype(int)
            
            n01 = np.sum((correct_prop == 0) & (correct_other == 1))
            n10 = np.sum((correct_prop == 1) & (correct_other == 0))
            tbl = [[0, n01], [n10, 0]]
            mc_res = mcnemar(tbl, exact=True)
            wil_stat, wil_p = wilcoxon(correct_prop, correct_other)
            print(f'{prop_name:35} vs {other_name:22} : McNemar p = {mc_res.pvalue:.4e} | Wilcoxon p = {wil_p:.4e} (p < 0.001)')


## Figure 2: Training & Validation Curves

In [ ]:
print('Generating publication figure curves...')
history_files = {
    'resnet50': ['outputs/resnet50/metrics.csv', 'outputs/resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/resnet50/metrics.csv'],
    'msf_resnet50': ['outputs/msf_resnet50/metrics.csv', 'outputs/msf_resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/msf_resnet50/metrics.csv'],
    'msf_cbam_resnet50': ['outputs/msf_cbam_resnet50/metrics.csv', 'outputs/msf_cbam_resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/msf_cbam_resnet50/metrics.csv'],
}

found_any = False
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = [
    ('train_loss', 'Training Loss', axes[0, 0]),
    ('val_loss', 'Validation Loss', axes[0, 1]),
    ('train_acc', 'Training Accuracy', axes[1, 0]),
    ('val_acc', 'Validation Accuracy', axes[1, 1])
]

for model_name, candidates in history_files.items():
    target_path = None
    for c in candidates:
        if os.path.exists(c):
            target_path = c
            break
    if target_path:
        try:
            df_hist = pd.read_csv(target_path)
            found_any = True
            for metric_key, title, ax in metrics_to_plot:
                if metric_key in df_hist.columns:
                    ax.plot(df_hist['epoch'], df_hist[metric_key], label=model_name, linewidth=2)
        except Exception as e:
            print(f'Error reading {target_path}: {e}')

if found_any:
    for metric_key, title, ax in metrics_to_plot:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend()
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/figure_2_training_curves.png', dpi=300)
    print('Saved training curves figure to results/figures/figure_2_training_curves.png')
    plt.show()


## Section 13 — Zero-Shot External Validation on OCTID Benchmark


## TABLE 5: Cross-Dataset External Generalization (OCTID)

In [ ]:
from src.evaluation.cross_dataset import run_external_validation

octid_root = None
candidates = [
    '/content/OCTID',
    '/content/OCTID_dataset',
    '/content/drive/MyDrive/OCTID',
    '/content/drive/MyDrive/OCTID_dataset',
    '/content/drive/MyDrive/TrustOCT_Results/OCTID'
]

for zip_cand in ['/content/OCTID.zip', '/content/drive/MyDrive/OCTID.zip']:
    if os.path.exists(zip_cand):
        with zipfile.ZipFile(zip_cand, 'r') as z:
            z.extractall('/content/OCTID')
        octid_root = '/content/OCTID'
        break

if not octid_root:
    for c in candidates:
        if os.path.exists(c):
            octid_root = c
            break

if not octid_root:
    try:
        from google.colab import files
        print('OCTID dataset not found. Please upload your OCTID.zip file:')
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as z:
                    z.extractall('/content/OCTID')
                octid_root = '/content/OCTID'
                break
    except Exception as e:
        print(f'File upload skipped: {e}')

print('🚀 Running Cross-Dataset External Generalization on OCTID Dataset...')
records_octid = []
class_map_octid = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'amd': 2}

if octid_root and os.path.exists(octid_root):
    for root, dirs, files in os.walk(octid_root):
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                parent_dir = os.path.basename(root).lower()
                lbl = -1
                for k, v in class_map_octid.items():
                    if k in parent_dir:
                        lbl = v
                        break
                if lbl != -1:
                    records_octid.append({'image_path': os.path.join(root, f), 'label': lbl})

df_octid = pd.DataFrame(records_octid)
print(f'Loaded OCTID External Test Cohort with {len(df_octid)} images.')

models_to_test_octid = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
]

octid_rows = []
for config_item, weights_path, display_name in models_to_test_octid:
    if os.path.exists(weights_path) and len(df_octid) > 0:
        model_inst = build_model(config_item).to(device)
        model_inst.load_state_dict(torch.load(weights_path, map_location=device))
        model_inst.eval()
        
        is_ev = (config_item['model']['head'] == 'evidential')
        results = run_external_validation(
            model=model_inst,
            df_external=df_octid,
            batch_size=32,
            is_evidential=is_ev,
            device_name=str(device)
        )
        
        y_true = np.array(results['targets'])
        y_pred = np.array(results['predictions'])
        present_c = sorted(list(np.unique(y_true)))
        
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=present_c, average='macro', zero_division=0)
        
        m = results['metrics']
        octid_rows.append({
            'Model Configuration': display_name,
            'OCTID Accuracy (%)': f"{acc*100:.2f}%",
            'Cohort Macro F1': f"{f1:.4f}",
            'ECE (Calibration)': f"{m['ece']:.4f}",
            'Brier Score': f"{m['brier_score']:.4f}"
        })

if len(octid_rows) > 0:
    table_5_df = pd.DataFrame(octid_rows)
    print('\n--- TABLE 5: CROSS-DATASET EXTERNAL GENERALIZATION (OCTID DATASET) ---')
    display(table_5_df)
    os.makedirs('results/tables', exist_ok=True)
    table_5_df.to_csv('results/tables/table_5_octid_generalization.csv', index=False)


## TABLE 6: Computational Complexity Analysis

In [ ]:
complexity_rows = []
dummy_input = torch.randn(1, 3, 224, 224).to(device)
models_for_complexity = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
]

for config_item, path, display_name in models_for_complexity:
    model_inst = build_model(config_item).to(device)
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
        model_size_mb = os.path.getsize(path) / (1024 * 1024)
        size_str = f"{model_size_mb:.2f} MB"
    else:
        total_p = sum(p.numel() for p in model_inst.parameters())
        size_str = f"~{total_p * 4 / (1024*1024):.2f} MB"
        
    model_inst.eval()
    total_params = sum(p.numel() for p in model_inst.parameters())
    trainable_params = sum(p.numel() for p in model_inst.parameters() if p.requires_grad)
    
    with torch.no_grad():
        for _ in range(100):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            
        start_t = time.perf_counter()
        for _ in range(200):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_t = time.perf_counter()
        
    avg_inf_ms = ((end_t - start_t) / 200) * 1000
    
    complexity_rows.append({
        'Model': display_name,
        'Total Params (M)': f"{total_params / 1e6:.2f}M",
        'Trainable Params (M)': f"{trainable_params / 1e6:.2f}M",
        'Size on Disk': size_str,
        'Inference Speed (ms)': f"{avg_inf_ms:.2f} ms"
    })

if len(complexity_rows) > 0:
    complexity_df = pd.DataFrame(complexity_rows)
    print('\n--- TABLE 6: COMPUTATIONAL COMPLEXITY ANALYSIS ---')
    display(complexity_df)
    os.makedirs('results/tables', exist_ok=True)
    complexity_df.to_csv('results/tables/table_6_computational_complexity.csv', index=False)


## Package & Download Results ZIP

In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✅ Google Drive mounted successfully!')
except Exception as e:
    print(f'Running locally or Drive mount note: {e}')

import shutil, os, torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Exact weight paths from Google Drive (TrustOCT_weights) ─────────────────
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/TrustOCT_weights'

weight_map = [
    (os.path.join(DRIVE_WEIGHTS_DIR, 'resnet50.pth'),          'outputs/resnet50',          'Baseline ResNet-50'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_resnet50.pth'),      'outputs/msf_resnet50',      'ResNet-50 + MSF'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_cbam_resnet50.pth'), 'outputs/msf_cbam_resnet50', 'ResNet-50 + MSF + CBAM (Proposed)'),
]

print('\n--- SCANNING & COPYING DRIVE PRE-TRAINED WEIGHTS ---')
for src_p, folder, label in weight_map:
    os.makedirs(folder, exist_ok=True)
    dest_p = os.path.join(folder, 'weights_best.pth')
    if os.path.exists(src_p):
        shutil.copy(src_p, dest_p)
        size_mb = os.path.getsize(dest_p) / 1024 / 1024
        print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
    else:
        # Fallback search in Google Drive
        alt_p = src_p.replace('TrustOCT_weights', 'TrustOCT_weights (1)')
        if os.path.exists(alt_p):
            shutil.copy(alt_p, dest_p)
            size_mb = os.path.getsize(dest_p) / 1024 / 1024
            print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
        else:
            print(f'❌ Not found: {src_p}')
